# 01 Setup GPU

This notebook checks the main Windows training environment.

It saves a simple report so we can prove which Python, CUDA, torch, Git, and compiler setup was used.

## Before You Run This Notebook

1. Open `x64 Native Tools Command Prompt for VS 2022`.
2. Activate the `fyp2` conda environment.
3. Start Jupyter from that same command window.

This matters because StyleGAN2-ADA needs the Visual Studio compiler in the active path when it tries to build CUDA plugins.

In [1]:
from pathlib import Path
import importlib
import json
import platform
import subprocess
import sys

# This small helper lets the notebook work whether it is opened from the repo root or from the notebooks folder.
def find_repo_root(start_path: Path) -> Path:
    if (start_path / "raw_data").exists():
        return start_path
    if start_path.name == "notebooks" and (start_path.parent / "raw_data").exists():
        return start_path.parent
    raise FileNotFoundError("Could not find the FYP2 repo root.")

REPO_ROOT = find_repo_root(Path.cwd())
RESULTS_DIR = REPO_ROOT / "results"
STYLEGAN_DIR = REPO_ROOT / "third_party" / "stylegan2-ada-pytorch"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# We check the common packages used across the training and comparison notebooks.
PACKAGE_IMPORTS = {
    "numpy": "numpy",
    "PIL": "PIL",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "torchvision": "torchvision",
    "timm": "timm",
    "scipy": "scipy",
}

print(f"Repo root: {REPO_ROOT}")
print(f"Results folder: {RESULTS_DIR}")
print(f"StyleGAN folder exists: {STYLEGAN_DIR.exists()}")

Repo root: C:\Users\FYP2610\Downloads\FYP2
Results folder: C:\Users\FYP2610\Downloads\FYP2\results
StyleGAN folder exists: True


In [2]:
# This helper runs a shell command and keeps both stdout and stderr for the saved report.
# If a command is missing, we save that as part of the report instead of crashing the notebook.
def run_command(command, cwd=None):
    try:
        result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
        return {
            "command": " ".join(command),
            "returncode": result.returncode,
            "stdout": result.stdout,
            "stderr": result.stderr,
        }
    except FileNotFoundError as error:
        return {
            "command": " ".join(command),
            "returncode": -1,
            "stdout": "",
            "stderr": str(error),
        }

# This print helper keeps the notebook output easy to read.
def print_result(name, result):
    print(f"\n{name}")
    print("-" * len(name))
    print(f"command: {result['command']}")
    print(f"returncode: {result['returncode']}")
    if result["stdout"].strip():
        print(result["stdout"].strip())
    if result["stderr"].strip():
        print(result["stderr"].strip())

In [3]:
# We check imports one by one so the report clearly shows what is installed and what is still missing.
package_results = {}
for package_name, import_name in PACKAGE_IMPORTS.items():
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "version_not_exposed")
        package_results[package_name] = {"status": "ok", "version": str(version)}
    except Exception as error:
        package_results[package_name] = {"status": "missing", "message": str(error)}

package_results

{'numpy': {'status': 'ok', 'version': '2.2.5'},
 'PIL': {'status': 'ok', 'version': '11.1.0'},
 'pandas': {'status': 'ok', 'version': '2.3.3'},
 'matplotlib': {'status': 'ok', 'version': '3.10.8'},
 'torch': {'status': 'ok', 'version': '2.11.0+cu128'},
 'torchvision': {'status': 'ok', 'version': '0.26.0+cu128'},
 'timm': {'status': 'ok', 'version': '1.0.26'},
 'scipy': {'status': 'ok', 'version': '1.15.3'}}

In [4]:
# These are the plain environment checks we want to keep for the paper record.
where_command = "where" if platform.system() == "Windows" else "which"
command_results = {
    "python_version": run_command([sys.executable, "--version"]),
    "git_version": run_command(["git", "--version"]),
    "compiler_path": run_command([where_command, "cl"]),
}

for name, result in command_results.items():
    print_result(name, result)


python_version
--------------
command: C:\Users\FYP2610\anaconda3\envs\fyp2\python.exe --version
returncode: 0
Python 3.10.20

git_version
-----------
command: git --version
returncode: 0
git version 2.53.0.windows.2

compiler_path
-------------
command: where cl
returncode: 0
C:\Program Files\Microsoft Visual Studio\2022\Community\VC\Tools\MSVC\14.43.34808\bin\Hostx64\x64\cl.exe


In [5]:
# Torch needs a separate check because we want to save the exact CUDA and GPU details.
torch_info = {}
try:
    import torch
    torch_info = {
        "torch_version": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "cuda_available": bool(torch.cuda.is_available()),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cuda_not_available",
    }
except Exception as error:
    torch_info = {"error": str(error)}

torch_info

{'torch_version': '2.11.0+cu128',
 'cuda_runtime': '12.8',
 'cuda_available': True,
 'gpu_name': 'NVIDIA GeForce RTX 4060'}

In [6]:
# We only run the StyleGAN smoke test if the vendored repo is already present.
stylegan_report = {
    "folder_exists": STYLEGAN_DIR.exists(),
    "train_help": None,
    "bias_act_test": None,
    "upfirdn2d_test": None,
    "bias_act_plugin_loaded": False,
    "upfirdn2d_plugin_loaded": False,
}

if STYLEGAN_DIR.exists():
    train_help_result = run_command([sys.executable, "train.py", "--help"], cwd=STYLEGAN_DIR)
    print_result("stylegan_train_help", train_help_result)
    stylegan_report["train_help"] = train_help_result

    bias_code = "import torch; from torch_utils.ops import bias_act; x=torch.randn([1,3,8,8], device='cuda'); y=bias_act.bias_act(x); print('bias_act_ok', y.shape)"
    bias_result = run_command([sys.executable, "-c", bias_code], cwd=STYLEGAN_DIR)
    print_result("stylegan_bias_act", bias_result)
    stylegan_report["bias_act_test"] = bias_result
    stylegan_report["bias_act_plugin_loaded"] = (
        bias_result["returncode"] == 0
        and "Falling back to slow reference implementation" not in (bias_result["stdout"] + bias_result["stderr"])
    )

    upfirdn_code = "import torch; from torch_utils.ops import upfirdn2d; x=torch.randn([1,1,8,8], device='cuda'); f=torch.tensor([1,3,3,1], dtype=torch.float32, device='cuda'); y=upfirdn2d.filter2d(x, f); print('upfirdn2d_ok', y.shape)"
    upfirdn_result = run_command([sys.executable, "-c", upfirdn_code], cwd=STYLEGAN_DIR)
    print_result("stylegan_upfirdn2d", upfirdn_result)
    stylegan_report["upfirdn2d_test"] = upfirdn_result
    stylegan_report["upfirdn2d_plugin_loaded"] = (
        upfirdn_result["returncode"] == 0
        and "Falling back to slow reference implementation" not in (upfirdn_result["stdout"] + upfirdn_result["stderr"])
    )
else:
    print("StyleGAN folder is not present yet. Skip the GAN smoke test for now.")

stylegan_report


stylegan_train_help
-------------------
command: C:\Users\FYP2610\anaconda3\envs\fyp2\python.exe train.py --help
returncode: 0
Usage: train.py [OPTIONS]

  Train a GAN using the techniques described in the paper "Training Generative
  Adversarial Networks with Limited Data".

  Examples:

  # Train with custom dataset using 1 GPU.
  python train.py --outdir=~/training-runs --data=~/mydataset.zip --gpus=1

  # Train class-conditional CIFAR-10 using 2 GPUs.
  python train.py --outdir=~/training-runs --data=~/datasets/cifar10.zip \
      --gpus=2 --cfg=cifar --cond=1

  # Transfer learn MetFaces from FFHQ using 4 GPUs.
  python train.py --outdir=~/training-runs --data=~/datasets/metfaces.zip \
      --gpus=4 --cfg=paper1024 --mirror=1 --resume=ffhq1024 --snap=10

  # Reproduce original StyleGAN2 config F.
  python train.py --outdir=~/training-runs --data=~/datasets/ffhq.zip \
      --gpus=8 --cfg=stylegan2 --mirror=1 --aug=noaug

  Base configs (--cfg):
    auto       Automatically selec

{'folder_exists': True,
 'train_help': {'command': 'C:\\Users\\FYP2610\\anaconda3\\envs\\fyp2\\python.exe train.py --help',
  'returncode': 0,
  'stdout': 'Usage: train.py [OPTIONS]\n\n  Train a GAN using the techniques described in the paper "Training Generative\n  Adversarial Networks with Limited Data".\n\n  Examples:\n\n  # Train with custom dataset using 1 GPU.\n  python train.py --outdir=~/training-runs --data=~/mydataset.zip --gpus=1\n\n  # Train class-conditional CIFAR-10 using 2 GPUs.\n  python train.py --outdir=~/training-runs --data=~/datasets/cifar10.zip \\\n      --gpus=2 --cfg=cifar --cond=1\n\n  # Transfer learn MetFaces from FFHQ using 4 GPUs.\n  python train.py --outdir=~/training-runs --data=~/datasets/metfaces.zip \\\n      --gpus=4 --cfg=paper1024 --mirror=1 --resume=ffhq1024 --snap=10\n\n  # Reproduce original StyleGAN2 config F.\n  python train.py --outdir=~/training-runs --data=~/datasets/ffhq.zip \\\n      --gpus=8 --cfg=stylegan2 --mirror=1 --aug=noaug\n\n  Bas

In [7]:
# The saved report makes it easy to compare machines and keep a clean record for the project log.
report = {
    "platform": platform.platform(),
    "python_executable": sys.executable,
    "package_results": package_results,
    "command_results": command_results,
    "torch_info": torch_info,
    "stylegan_report": stylegan_report,
}

output_path = RESULTS_DIR / "setup_gpu_report.json"
output_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Saved setup report to: {output_path}")

Saved setup report to: C:\Users\FYP2610\Downloads\FYP2\results\setup_gpu_report.json
